# L2G prioritised genes

For each credible set, the genes L2G nominates: every gene with score >= 0.5, or, if none
reaches that, the single top gene if its score >= 0.1. Each nomination carries the four
direct reasons a gene could be picked (eQTL colocalisation, pQTL colocalisation, PAV,
nearest TSS). Methods "Dynamic training set and L2G model training".

Writes `prioritised_genes_per_cs`, `study_annotation`, `prioritised_genes_annotated`,
`prioritised_genes_diseases`, `prioritised_genes_measurements`.

In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from pyspark.sql import Window
from pyspark.sql import functions as f

from manuscript_methods import discovery, paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 00:27:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Nominated genes per credible set

In [2]:
CLPP, H4, VEP_PAV = 0.01, 0.8, 0.66

l2g = session.spark.read.parquet(str(paper.ROOT / "data/25.06/irene_1208_l2g_predictions")).select(
    "studyLocusId", "geneId", "score"
)

features = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .filter(f.col("isProteinCoding") == 1)
    .select(
        "studyLocusId",
        "geneId",
        f.when((f.col("eQtlColocClppMaximum") >= CLPP) | (f.col("eQtlColocH4Maximum") >= H4), 1)
        .otherwise(0)
        .alias("eQTL_coloc"),
        f.when((f.col("pQtlColocClppMaximum") >= CLPP) | (f.col("pQtlColocH4Maximum") >= H4), 1)
        .otherwise(0)
        .alias("pQTL_coloc"),
        f.when(f.col("vepMaximum") >= VEP_PAV, 1).otherwise(0).alias("VEP"),
        f.when(f.col("distanceSentinelTssNeighbourhood") == 1, 1).otherwise(0).alias("distanceTSS"),
    )
)
print("protein-coding CS-gene pairs scored:", features.count())

protein-coding CS-gene pairs scored: 10623371


In [3]:
confident = l2g.filter(f.col("score") >= 0.5)
# For credible sets where nothing reaches 0.5, keep the single best gene if it reaches 0.1.
fallback = (
    l2g.join(confident.select("studyLocusId").distinct(), "studyLocusId", "left_anti")
    .withColumn("rank", f.row_number().over(Window.partitionBy("studyLocusId").orderBy(f.desc("score"))))
    .filter((f.col("rank") == 1) & (f.col("score") >= 0.1))
    .drop("rank")
)
prioritised = confident.unionByName(fallback)

per_cs = features.join(prioritised.drop("score"), on=["studyLocusId", "geneId"], how="inner")
per_cs.write.mode("overwrite").parquet(paper.derived("prioritised_genes_per_cs"))

per_cs = session.spark.read.parquet(paper.derived("prioritised_genes_per_cs"))
print("CS-gene prioritisations:", per_cs.count())
print("credible sets with a gene:", per_cs.select("studyLocusId").distinct().count())

CS-gene prioritisations: 788767


credible sets with a gene: 765893


## Study annotation: year, ancestry class, effective sample size

In [4]:
# FinnGen R12 carries no publication date in the release; it is pinned to the R12 release date.
studies = (
    StudyIndex.from_parquet(session, paper.release("study"))
    .df.filter(f.col("studyType") == "gwas")
    .withColumn(
        "publicationDate",
        f.when(f.col("projectId") == "FINNGEN_R12", f.lit("2024-11-04")).otherwise(f.col("publicationDate")),
    )
    .withColumn("year", f.col("publicationDate").substr(1, 4).cast("int"))
    .withColumn(
        "effectiveSampleSize",
        4.0 * f.col("nCases") * f.col("nControls") / (f.col("nCases") + f.col("nControls")),
    )
    .transform(discovery.classify_ancestry)
)

annotation = studies.select(
    "studyId",
    "projectId",
    "traitFromSource",
    "year",
    "publicationDate",
    "nSamples",
    "nCases",
    "nControls",
    "effectiveSampleSize",
    "diseaseIds",
    "nLdPopulations",
    "ldStructureNote",
    "predominantAncestry",
    "predominantFraction",
    "nfeFraction",
    "ancestryClass",
)
annotation.write.mode("overwrite").parquet(paper.derived("study_annotation"))
annotation = session.spark.read.parquet(paper.derived("study_annotation")).cache()
annotation.groupBy("ancestryClass").count().show()

+-------------+-----+
|ancestryClass|count|
+-------------+-----+
|        mixed|11725|
|          EUR|65253|
|      non-EUR|23548|
+-------------+-----+



## Nominations annotated with MAF, effect size, year and ancestry

In [5]:
lead = session.spark.read.parquet(paper.derived("lead_variant_effect")).select(
    "studyId",
    "studyLocusId",
    "variantId",
    f.abs(f.col("rescaledStatistics.absEstimatedBeta")).alias("absBeta"),
    f.col("majorLdPopulationMaf.value").alias("maf"),
)

annotated = (
    per_cs.join(lead, on="studyLocusId", how="inner")
    .join(annotation, on="studyId", how="inner")
    # A null MAF is treated as rare.
    .fillna({"maf": 0})
    .withColumn("freqClass", f.when(f.col("maf") >= discovery.MAF_COMMON, "common").otherwise("rare"))
)
annotated.write.mode("overwrite").parquet(paper.derived("prioritised_genes_annotated"))

annotated = session.spark.read.parquet(paper.derived("prioritised_genes_annotated"))
print("gene x credible-set rows:", annotated.count())

26/08/19 00:27:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


gene x credible-set rows: 788767


## Disease and measurement subsets, restricted to qualifying credible sets

In [6]:
for name, qualifying in [
    ("prioritised_genes_diseases", "qualifying_credible_sets"),
    ("prioritised_genes_measurements", "qualifying_measurement_credible_sets"),
]:
    cs = session.spark.read.parquet(paper.derived(qualifying)).select("studyLocusId").distinct()
    annotated.join(cs, "studyLocusId", "inner").write.mode("overwrite").parquet(paper.derived(name))
    subset = session.spark.read.parquet(paper.derived(name))
    print(name, "rows:", subset.count(), "genes:", subset.select("geneId").distinct().count())

prioritised_genes_diseases rows: 70400 genes: 8285


prioritised_genes_measurements rows: 453009 genes: 15160


## Cross-check against the pre-refactor tables

In [7]:
print(
    "baseline prioritised genes per CS:",
    session.spark.read.parquet(paper.baseline("list_of_prioritised_genes_per_CS.parquet")).count(),
)
print("baseline enrichment set:", session.spark.read.parquet(paper.baseline("l2g_full_for_enrichment")).count())

baseline prioritised genes per CS: 788767
baseline enrichment set: 70400
